> 在线讲义：[第 12 章 · Pre-LN：为什么 Post-LN 离不开 warmup，把 LayerNorm 挪个位置就好了](https://wikieden.github.io/beyond-transformer/lessons/12-pre-ln.html) · 本书《超越 Transformer》

# 第 12 章 · Pre-LN：为什么 Post-LN 离不开 warmup，把 LayerNorm 挪个位置就好了

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/12-pre-ln.html) · 仓库里 `beyond-transformer/notes/lessons/12-pre-ln.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 各层梯度 | 手写反向 | `grad_norms("post")`、`grad_norms("pre")` | Post 约 250；Pre 262 → 102 |
| 第二步 随层数 | Theorem 1 | `last_layer_vs_depth()` | Pre × √L ≈ 350 |

Post-LN 的梯度集中在靠近输出的层、且不随层数变小，Pre-LN 的梯度随层数变小、各层更均匀。
在真实模型上训练 Post-LN 和 Pre-LN 并比较曲线是 a1 的 pre_norm_ablation 和里程碑 T-M4 的内容，这里只看初始化时的梯度。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `ln_fwd`**：小工具。

In [ ]:
def ln_fwd(x):
    mu = x.mean(-1, keepdims=True)
    sd = np.sqrt(((x - mu) ** 2).mean(-1, keepdims=True) + 1e-6)
    return (x - mu) / sd, sd

**函数 `ln_bwd`**：y = (x − μ)/σ 的反向：dx = (dy − mean(dy) − y·mean(dy·y)) / σ。

In [ ]:
def ln_bwd(dy, y, sd):
    """y = (x − μ)/σ 的反向：dx = (dy − mean(dy) − y·mean(dy·y)) / σ。"""
    return (dy - dy.mean(-1, keepdims=True) - y * (dy * y).mean(-1, keepdims=True)) / sd

**函数 `grad_norms`**：每层子层 f_l(x) = x W_lᵀ（W ~ N(0, 1/d)）。Post：x ← LN(x + f(x))；Pre：x ← x + f(LN(x))，最后再 LN。

In [ ]:
def grad_norms(kind: str, L: int = 12, d: int = 64, batch: int = 32, seed: int = 0):
    """每层子层 f_l(x) = x W_lᵀ（W ~ N(0, 1/d)）。Post：x ← LN(x + f(x))；Pre：x ← x + f(LN(x))，最后再 LN。
    loss = ½‖输出 − 目标‖²（目标随机）。返回每层 W 的梯度 Frobenius 范数。"""
    rng = np.random.default_rng(seed)
    W = [rng.normal(0, 1 / np.sqrt(d), (d, d)) for _ in range(L)]
    x = rng.normal(size=(batch, d))
    target = rng.normal(size=(batch, d))
    cache = []
    for w in W:
        if kind == "post":
            z = x + x @ w.T
            y, sd = ln_fwd(z)
            cache.append((x, z, y, sd))
            x = y
        else:
            u, sd = ln_fwd(x)
            cache.append((x, u, sd))
            x = x + u @ w.T
    if kind == "pre":
        out, sdf = ln_fwd(x)
        g = ln_bwd(out - target, out, sdf)
    else:
        g = x - target
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        if kind == "post":
            xin, z, y, sd = cache[l]
            dz = ln_bwd(g, y, sd)
            grads[l] = float(np.linalg.norm(dz.T @ xin))
            g = dz + dz @ W[l]
        else:
            xin, u, sd = cache[l]
            grads[l] = float(np.linalg.norm(g.T @ u))
            g = g + ln_bwd(g @ W[l], u, sd)
    return grads

**函数 `last_layer_vs_depth`**：小工具。

In [ ]:
def last_layer_vs_depth(depths=(6, 12, 24, 48)):
    return {L: (grad_norms("post", L)[-1], grad_norms("pre", L)[-1]) for L in depths}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 两种放法（初始化时、12 层、d = 64）各层 W 的梯度范数")
post, pre = grad_norms("post"), grad_norms("pre")
for l in (0, 3, 6, 9, 11):
    print(f"  第 {l + 1:2d} 层：Post-LN {post[l]:7.3f}   Pre-LN {pre[l]:7.3f}")
print(f"  最后一层 / 第一层：Post-LN {post[-1] / post[0]:.2f}，Pre-LN {pre[-1] / pre[0]:.2f}")
print("\n第二步 最后一层的梯度随层数 L 怎么变")
for L, (a, b) in last_layer_vs_depth().items():
    print(f"  L = {L:2d}：Post-LN {a:7.3f}   Pre-LN {b:7.3f}   Pre-LN × √L = {b * np.sqrt(L):7.3f}")

## 3. 代码里才看得到的

- **Post-LN 的反向每层都要穿过一个 LayerNorm**：梯度每层都被“去均值、去投影、除以 σ”，这就是它和层数无关的来源；Pre-LN 的主干梯度可以直接沿残差一路传回去。
- **只看初始化时**：训练起来以后参数变了，梯度分布也会变；在真实模型上训练、比曲线是 a1 的 pre_norm_ablation 和里程碑 T-M4。

## 4. 自己改一改

- 把 `grad_norms` 的 d 改成 256，两种放法的梯度量级怎么变？
- 把子层的初始化标准差从 1/√d 改成 2/√d，Post-LN 和 Pre-LN 谁更敏感？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 质检站在工序后，Post-LN 每层都整理；站在工序前，Pre-LN 主干原样流。
> Post-LN 输出附近梯度大，和层数无关，不 warmup 就炸；
> Pre-LN 梯度按根号 L 变小，可以不 warmup。
> 主干越加越大，最后补一个 norm；后来的变体都把 norm 放子层两头。